In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, train_test_split
from tqdm import tqdm
import glob
import matplotlib.pyplot as plt
import copy
import random
from collections import Counter

from sklearn.cluster import KMeans
from sklearn.decomposition import KernelPCA
from utils import RandomForestClassifierUnc, XGBEnsemble
from xgboost import XGBClassifier

sns.set_style("whitegrid")

import matplotlib

# set font size to 16
matplotlib.rcParams.update({"font.size": 16})

ROOT = "../CIC2019/"
SEED = 4

In [7]:
files_cic2019 = glob.glob(ROOT + "03-11/*.csv")
files_cic2019.sort()
files_cic2019

['../CIC2019/03-11/01_Portmap.csv',
 '../CIC2019/03-11/03_LDAP.csv',
 '../CIC2019/03-11/04_MSSQL.csv',
 '../CIC2019/03-11/05_UDP.csv',
 '../CIC2019/03-11/06_UDPLag.csv',
 '../CIC2019/03-11/07_Syn.csv']

In [ ]:
#Trim the dataset: take 300 benign and 300 malicious samples from each file
dfs = []
constraint = 3000 
for file_id, file in enumerate(tqdm(files_cic2019)):
    df = pd.read_csv(file, engine="pyarrow")
    df.columns = df.columns.str.strip()
    benign_df = df[df["Label"] == "BENIGN"]
    if len(benign_df) > constraint:
        benign_df = benign_df.sample(n=constraint)
    malicious_df = df[df["Label"] != "BENIGN"]
    malicous_sampled_df = malicious_df.sample(n=len(benign_df))
    #malicous_sampled_df = malicious_df.sample(n=2 * len(benign_df), random_state=SEED)
    undersampled_df = pd.concat([malicous_sampled_df, benign_df])
    undersampled_df["task_id"] = file_id
    dfs.append(undersampled_df)
df = pd.concat(dfs)

100%|██████████| 6/6 [00:11<00:00,  1.95s/it]


In [9]:
df_dropped = df.drop(
    columns=[
        "Unnamed: 0",
        "Flow ID",
        "Source IP",
        "Destination IP",
        "Protocol",
        "Source Port",
        "Destination Port",
        "Timestamp",
        "SimillarHTTP",
        "Inbound",
    ]
)

In [ ]:
# replace infinities with -1
df_dropped = df_dropped.replace([np.inf, -np.inf], -1)
# replace nans with -2
df_dropped = df_dropped.fillna(-2)

df_dropped.to_csv("../CIC_dfdropped/03-11/df_dopped_3000.csv", index=False)